# ಪಾಠ 18: ಕ್ರಿಪ್ಟೋಗ್ರಾಫಿಕ್ ರಸೀದಿಗಳೊಂದಿಗೆ AI ಏಜೆಂಟ್‌ಗಳನ್ನು ಸುರಕ್ಷಿತ ಮಾಡುವುದು  

## ಹ್ಯಾಂಡ್ಸ್-ಆನ್ ನೋಟ್ಬುಕ್  

ಈ ನೋಟ್ಬುಕ್ ನಾಲ್ಕು ಅಭ್ಯಾಸಗಳನ್ನು ನಡೆಸುತ್ತದೆ:  

1. ಏಜೆಂಟ್ ಟೂಲ್ ಕರೆಗಾಗಿ ನಿಮ್ಮ ಪ್ರಥಮ ರಸೀದಿಯನ್ನು **ಸೈನ್ ಮಾಡಿ** ಮತ್ತು ಅದನ್ನು ಪರಿಶೀಲಿಸಿ.  
2. ರಸೀದಿಯನ್ನು **ತಿದ್ದುಪಡಿ ಮಾಡಿ** ಮತ್ತು ಪರಿಶೀಲನೆ ವಿಫಲವಾಗುವುದನ್ನು ನೋಡಿ.  
3. **ಮೂರು ರಸೀದಿ ಸರಪಳಿಯನ್ನು ನಿರ್ಮಿಸಿ** ಮತ್ತು ಸರಪಳಿಯ ಸಮಗ್ರತೆಯನ್ನು ದೃಢೀಕರಿಸಿ.  
4. ಪ್ರತಿ ಕ್ರಿಯೆಯೂ ರಸೀದಿ ಉತ್ಪಾದಿಸುವಂತೆ **ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್‌ವರ್ಕ್ ಟೂಲ್ ಕರೆಗಳನ್ನು ಸುರುಳಿ ಹಾಕಿ**.  

ಎಲ್ಲಾ ಕ್ರಿಪ್ಟೋಗ್ರಾಫಿಕ್ ಮೂಲಭೂತಗಳನ್ನು ಉತ್ತಮವಾಗಿ ನಿರ್ವಹಿಸಲಾದ ಲೈಬ್ರರಿಗಳಿಂದ ಆಮದು ಮಾಡಲಾಗಿದೆ (`pynacl` Ed25519 ಗೆ, `jcs` RFC 8785 ಕ್ಯಾನೊನಿಕಲ್ JSON ಗೆ, `hashlib` Python ಮಾನಕ ಲೈಬ್ರರಿಯಿಂದ SHA-256 ಗೆ). ರಸೀದಿ ಲಾಜಿಕ್ ಸ್ವತಃ ಓದಲು ಮತ್ತು ಬದಲಾಯಿಸಲು ಸಾಧ್ಯವಿರುವ ಸ್ಪಷ್ಟ ಪೈಥಾನ್.  

ಸೆಲ್‌ಗಳನ್ನು ಕ್ರಮವಾಗಿ ರನ್ ಮಾಡಿ. ಪ್ರತಿಯೊಂದು ವಿಭಾಗ ಕೂಡಿಲು ಮತ್ತು ಸ್ವಯಂಸಂಕೀರ್ಣವಾಗಿದೆ.  


## ಸೆಟಪ್

ಎರಡು ಅವಶ್ಯಕತೆಗಳನ್ನು ಸ್ಥಾಪಿಸಿ. ಎರಡಕ್ಕೂ ಅನುಮತಿಸಬಹುದಾದ ಪರವಾನಗಿಗಳು ಇವೆ (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## ಸಹಾಯಕ ಉಪಕರಣಗಳು

ಈ ಎರಡು ಸಹಾಯಕಗಳು ಬೇಸ್64ಯುಆರ್ ಸಂಕೇತೀಕರಣವನ್ನು (ಪ್ಯಾಡಿಂಗ್ ಇಲ್ಲದೆ) ಮತ್ತು任意 ವಸ್ತುಗಳ SHA-256 ಹ್ಯಾಶಿಂಗ್ ಅನ್ನು ನಿರ್ವಹಿಸುತ್ತವೆ. ಅವು ಉಳಿದ ನೋಟ್‌ಬುಕ್ ಅನ್ನು ರಸೀದಿ ಲಾಜಿಕ್ ಮೇಲೆ ಕೇಂದ್ರೀಕರಿಸುತ್ತದೆ.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## ವಿಭಾಗ 1: ನಿಮ್ಮ ಮೊದಲ ರಸೀದಿಯನ್ನು ಸಹಿ ಮಾಡಿ

ನಮ್ಮ **ಕಾಂಟೋಸೋ ಟ್ರಾವೆಲ್** ಏಜೆಂಟ್ ಸಿಡ್ನಿ ಯಿಂದ ಲಾಸ್ ಏಂಜಲಿಸ್ ಗೆ ವಿಮಾನಗಳನ್ನು ಹುಡುಕಿದ್ದಾರೆಂದು ಊಹಿಸಿಕೊಳ್ಳಿ. ನಾವು ಈ ಉಪಕರಣ ಕರೆ ಬಗ್ಗೆ ಗೇಟ್‌ವೇ ನ ಸುಳಿವು ದಾಖಲಿಸಲು ಬಯಸುತ್ತೇವೆ, ಭವಿಷ್ಯದ ಪರೀಕ್ಷಕ ಅದನ್ನು ಬೇರೆ ಪ್ರತ್ಯೇಕ ಇಷ್ಯೂಯರ್ ಕೀಲಿಯಡಿ ಪರಿಶೀಲಿಸಬಹುದು.

### ಹಂತ 1.1: ಸಹಿ ಹಾಕುವ ಕೀಲಿಯನ್ನು ಉತ್ಪಾದಿಸಿ

ಉತ್ಪಾದನೆಯಲ್ಲಿ, ಏಜೆಂಟ್‌ ನ ಸಹಿ ಹಾಕುವ ಕೀಲಿಯು ಹಾರ್ಡ್‌ವೇರ್ ಸೆಕ್ಯುರಿಟಿ ಮoduಲ್ (HSM), ಅಜೂರ್ ಕೀಲಿಮನೆ ಅಥವಾ ಅದೇ ರೀತಿ ರಕ್ಷಿತ ಸಂಗ್ರಹಣದಲ್ಲಿ ಇರುತ್ತದೆ. ಈ ಪಾಠದಲ್ಲಿ ನಾವು ಮೆಮರಿಯಲ್ಲಿ ಹೊಸ ಕೀಲಿಯನ್ನು ರಚಿಸುತ್ತೇವೆ.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### ಹಂತ 1.2: ರಸೀದಿ ಪೇಲೋಡ್ ನಿರ್ಮಿಸಿ

ಪೇಲೋಡ್‌ನಲ್ಲಿ ನಾವು ರಸೀದಿ ದೃಢೀಕರಿಸಬೇಕಾದ ಎಲ್ಲದರಿರುತ್ತದೆ: ಯಾರು ಮಾಡಿದರು, ಯಾವ ಉಪಕರಣ, ಯಾವ ಆರ್ಗ್ಯುಮೆಂಟ್‌ಗಳೊಂದಿಗೆ, ಏನು ಹಿನ್ನೆಲೆ ಬಂತು, ಯಾವ ನೀತಿಯಡಿ, ಮತ್ತು ಯಾವಾಗ. ರಸೀದಿ ಸಂವೇದನಶೀಲ ವಸ್ತುಗಳನ್ನು ಹಿಗ್ಗಿಸದಂತೆ, ಆರ್ಗ್ಯುಮೆಂಟ್‌ಗಳ ಮತ್ತು ಫಲಿತಾಂಶದ ಹ್ಯಾಶ್ ಮಾಡುತ್ತೇವೆ, ಅವುಗಳನ್ನು ನೇರವಾಗಿ ಸೇರಿಸುವ ಬದಲು.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### ಹಂತ 1.3: ಸ್ವೀಕೃತಿಯನ್ನು ಸಹಿ ಮಾಡಿ ಮತ್ತು ಸಂಯೋಜಿಸಿ

ಮೂರು ಹಂತಗಳು:

1. JCS ಬಳಸಿ ಪೇಲೋಡ್ ಅನ್ನು ಕ್ಯಾನೋನೈಜ್ ಮಾಡಿ ಆದ್ದರಿಂದ ಒಂದೇ ತಾರ್ಕಿಕ ಸ್ವೀಕೃತಿ ಉತ್ಪಾದಿಸುವ ಎರಡು ಅನುಷ್ಠಾನಗಳು ಬೈಟ್-ಒಂದುತರಣೆಯ ಬೈಟ್‌ಗಳನ್ನು ಉತ್ಪಾದಿಸುತ್ತವೆ.
2. Ed25519 ಖಾಸಗಿ ಕೀಲಿಯನ್ನು ನೇರವಾಗಿ ಕ್ಯಾನೋನಿಕಲ್ ಬೈಟ್‌ಗಳ ಮೇಲೆ ಸಹಿ ಮಾಡು. PureEdDSA ಸಂದೇಶವನ್ನು ಒಳಗೊಂಡಂತೆ ಹ್ಯಾಶ್ ಮಾಡುತ್ತದೆ, ಆದ್ದರಿಂದ ಹೆಚ್ಚುವರಿ ಪ್ರೀ-ಹ್ಯಾಶ್ ಪ್ರೋಟೋಕಾಲ್ ಅನ್ನು ಬದಲಾಯಿಸುತ್ತದೆ.

ಸಹಿ ನಂತರ ಅಂತಿಮ ಸ್ವೀಕೃತಿ ಉತ್ಪಾದಿಸಲು ಮೂಲ ಪೇಲೋಡ್‌ಗೆ ಜೋಡಿಸಲಾಗುತ್ತದೆ.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### ಹಂತ 1.4: ರಸೀದಿ ಪರಿಶೀಲನೆ ಮಾಡಿ

ಪರಿಶೀಲನೆ ಪ್ರಕ್ರಿಯೆಯನ್ನು ಹಿಂತಿರುಗಿಸುತ್ತದೆ. ನಾವು ಸಹಿಯನ್ನು ತೆಗೆದುಹಾಕಿ, ನೈಸರ್ಗಿಕ ಬೈಟ್ಗಳನ್ನು ಮರುಳಿಸಲು, ಮತ್ತು ಸಹಿಯನ್ನು ಪ್ರಕಾಶಕರ ಸಾರ್ವಜನಿಕ ಕೀಲಿಗೆ ವಿರುದ್ಧ ಪರಿಶೀಲಿಸುತ್ತೇವೆ.

ಪರಿಶೀಲನೆ ಆಫ್‌ಲೈನ್ ಆಗಿದೆ: ಕರೆ ಮಾಡುವ ಸೇವೆಯೂ ಇಲ್ಲ ಮತ್ತು ನೆಟ್‌ವರ್ಕ್ ಪ್ರವೇಶವೂ ಇಲ್ಲ. ಅದು ಇನ್ನೂ ಒಂದು ವಸ್ತು ಬೇಕಾಗುತ್ತದೆ ರಸೀದಿಯ ಹೊರತಾಗಿ: ಪ್ರಕಾಶಕರ ಸಾರ್ವಜನಿಕ ಕೀಲಿ, ಇದು ಯಾವುದಾದರೂ ಜೇವಿಕೆ ಮೂಲಕ ಪಡೆಯಬಹುದು, ಉದಾಹರಣೆಗೆ ಪ್ರಕಾಶಕರು ಪ್ರಕಟಿಸುವ JWK ಸೆಟ್ ಅಥವಾ ಮುಂಚಿತವಾಗಿ ಪಿನ್ ಮಾಡಲಾದ ಮೌಲ್ಯ. ರಸೀದಿಯೊಳಗಿನ `public_key` ಕ್ಷೇತ್ರ ಆ ಪಾತ್ರ ವಹಿಸಲು ಸಾಧ್ಯವಿಲ್ಲ, ಏಕೆಂದರೆ ಯಾರು ರಸೀದಿ ಬರೆಯುತ್ತಾರೋ ಅವರು ಆ ಕ್ಷೇತ್ರವನ್ನೂ ಬರೆಯುತ್ತಾರೆ. ವಿಭಾಗ 2 ಈ ನಕಲಿ ತಪ್ಪಿಸಲು ಸಹಾಯ ಮಾಡುತ್ತದೆ.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


ನೀವು `Receipt is valid: True`, `Pre-hashed receipt valid: False` ಮತ್ತು `Malformed receipts rejected: True` ಅನ್ನು ನೋಡಬೇಕು. ಸಕಾರಾತ್ಮಕ ಕೇಸ್ ನೇರ-JCS ಸಹಿಯನ್ನು ಮಾರ್ಗ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ ಎಂದು ಸಲ್ಲಿಸುತ್ತದೆ; ನಕಾರಾತ್ಮಕ ನಿಯಂತ್ರಣ ಸಹಿ-ವಿಸ್ತಾರ ನಿಯಮವನ್ನು ಪ್ರಾಯೋಗಿಕವಾಗಿ ಮಾಡುತ್ತದೆ ಬದಲಿಗೆ ಅದನ್ನು ಪ್ರೋಸ್ ಆಗಿ ಬಿಟ್ಟಿರುವುದನ್ನು; ಕೊನೆಯ ಸಾಲಿನಲ್ಲಿ ದೋಷಮಾನ ಇನ್‌ಪುಟ್ ಅನ್ನು ತೋರುತ್ತದೆ `False` ಅನ್ನು ಎಸೆಪ್ಟ್ ಗಳಿಸುವ ಬದಲಿಗೆ ಮೀರಿಸಿದರೆ.


## ವಿಭಾಗ 2: ರಸೀದಿಯನ್ನು ತಡವಾಗಿ ಮಾಡುವುದು

ರಸೀದಿಗಳ ಸಂಪೂರ್ಣ ಉದ್ದೇಶವು ಅವು ತಡವಾಗಿ ಕಾಣುವಂತೆ ಮಾಡುವುದು. ನಾವು ಅದನ್ನು ಸಲ್ಲಿಸೋಣ.

ನಾವು ರಸೀದಿಯ ಸ್ಪಷ್ಟವಾಗಿ ಒಂದು ಅಕ್ಷರವನ್ನು ಬದಲಾಯಿಸಿ ತಪಾಸಣೆಯ ವಿಫಲವಾಗುವಿಕೆಯನ್ನು ನೋಡುತ್ತೇವೆ.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### ಏನು ನೇರವಾಗಿದ್ದೇ?

ನಾವು `policy_id` ಅನ್ನು ಬದಲಿಸಿದಾಗ, ಕ್ಯಾನ್ಯಾನಿಕಲ್ ಬೈಟ್ಸ್ ಬದಲಾಗಿದೆ. ಆಹಿತ ನಕಲಿ (ಅದ್ದು ಮೊದಲಿನ ಕ್ಯಾನ್ಯಾನಿಕಲ್ ಬೈಟ್ಸ್ ಮೇಲೆ ಇತ್ತು) ಈಗ ಹೊಂದಿಕೆಯಾಗುವುದಿಲ್ಲ. ಪರಿಶೀಲನೆ ಸರಿಯಾಗಿ `False` ಅನ್ನು ತಿರುಗಿಸುತ್ತದೆ.

ಸಹಿ ಹಾಕಲಾದ ಕ್ಷೇತ್ರವನ್ನು, ಅಂದರೆ `signature` ವಸ್ತುವಿನ ಹೊರಗಿನ ಏನಾದರೂ ಬದಲಿಸುವುದು ಸಾಧ್ಯವಿಲ್ಲ, ಮತ್ತು ಇನ್ನೂ ಪ್ರಾಪ್ತಿಯನ್ನು ಇಸ್ಯೂಯರ್‌ನ ಸಾರ್ವಜನಿಕ ಕೀಲಿಗೆ ವಿರುದ್ಧ ಪರಿಶೀಲಿಸಲು, ಆಕ್ರಮಣಕಾರನು ಇಸ್ಯೂಯರ್‌ನ ಖಾಸಗಿನ ಕೀಲಿಯನ್ನು ಹೊಂದಿಲ್ಲದಿದ್ದರೆ ಹೊರತು. `signature` ವಸ್ತುವು ಸ್ವತಃ ಸಹಿ ಹಾಕಲ್ಪಟ್ಟಿಲ್ಲ: `sig` ಸಹಿಯಾಗಿದೆ, ಮತ್ತು `verify_receipt` `public_key` ಅನ್ನು ನಿರ್ಲಕ್ಷಿಸುತ್ತದೆ, ಆದ್ದರಿಂದ ಆ ಕ್ಷೇತ್ರವನ್ನು ಮಾತ್ರ ಸಂಪಾದಿಸುವುದರಿಂದ ಏನೂ ಬದಲಾಗುವುದಿಲ್ಲ. ಖಾಸಗಿ ಕೀ ಕೀವಾಲ್ಟ್ ನಲ್ಲಿ ಇದ್ದಾರೆ ಮತ್ತು ಪರಿಶೀಲಕರು ಇಸ್ಯೂಯರ್‌ನ ಸ್ವತಂತ್ರವಾಗಿ ಪ್ರಕಟಿಸಿದ ಕೀಲಿಯನ್ನು ಬಳಸುತ್ತಿದ್ರೆ, ಸಹಿ ಹಾಕಲಾದ ಕ್ಷೇತ್ರದಲ್ಲಿ ಬದಲಾವಣೆಗಳನ್ನು ಮರೆಮಾಚುವುದು ಸಾಧ್ಯವಿಲ್ಲ. ಆ ಪರಿಸ್ಥಿತಿ ಮಹತ್ವದದ್ದು: ಮುಂದಿನ ಸೆಲ್ ವಿಭಿನ್ನ ಕೀಲಿಯಿಂದ ಪರಿಪೂರ್ಣವಾಗಿ ಪರಿಶೀಲನೆಯಾದ ಬದಲಿಸಿದ ರಸೀದಿಯನ್ನು ತೋರಿಸುತ್ತದೆ.

ನೀವು ಸ್ವತಃ ಪ್ರಯತ್ನಿಸಿ: ಮೇಲಿನ ಸೆಲ್‌ನಲ್ಲಿನ `tool_name` ಅಥವಾ `agent_id` ಅಥವಾ `timestamp` ಅನ್ನು ಬದಲಿಸಿ ಮತ್ತು ಮರು ಚಾಲನೆ ಮಾಡಿ. ಸಹಿಯಿಂದಾಗಿ ಯಾವುದೇ ಬದಲಾವಣೆ ಅಮಾನ್ಯ ರಸೀದಿಯನ್ನು ಉತ್ಪಾದಿಸುತ್ತದೆ.


### ಸಹಿಯನ್ನು ಕಾಣದ ತಿದ್ದುಪಡಿಯಲ್ಲಿ

ಮೇಲಿನ ಹಾಳುಗಾರರು ಪುನಃ ಸಹಿ ಹಾಕಲಿಲ್ಲ. ಹೆಚ್ಚು ಎಚ್ಚರಿಕೆಯುಳ್ಳವರು: ರಸೀದಿ ಬದಲಾಯಿಸುತಿದ್ದರು,
ತಾವು ಸ್ವತಃ ರಚಿಸಿದ ಕೀ ಮೆಚ್ಚಿಸಿ, ಆ ಕೀಯನ್ನು ಅದರಲ್ಲಿ ಇರಿಸಿ
`public_key` ಕ್ಷೇತ್ರ. ಫಲಿತಾಂಶವು ಸಹಿ ಸಂಪೂರ್ಣವಾಗಿ ವಾಸ್ತವಿಕವಾದ ಸ್ವೀಕೃತಿಯಾಗಿದೆ
ಅದರ ಒಳ ವಿಷಯಗಳಿಗೆ.

ಇದನ್ನು ಓಡಿಸಿ ಮತ್ತು ಎರಡಲ್ಲದ ಯಾವುದಾದರೂ ಪರಿಶೀಲನೆ ಅದನ್ನು ಹಿಡಿದೆಯೇ ನೋಡಿರಿ.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


ನೀವು `True` ನಂತರ `False` ಅನ್ನು ನೋಡಬೇಕು. ಈ ಮೂರು `assert` ಸಾಲುಗಳು ಅದನ್ನು
ಒಂದು ಪರಿಶೀಲನೆಯಾಗಿಸುತ್ತದೆ: forgery ಆಗಾಗ issuerನ ಕೀಲಿಗೆ ವಿರುದ್ಧವಾಗಿ ಪರಿಶೀಲಿಸಿದರೆ ಸೆಲ್ ಒಂದು ದೋಷವನ್ನು ಎತ್ತುತ್ತದೆ.


forgery ಆಂತರಿಕವಾಗಿ ಸुसಂಗತವಾಗಿದೆ. ಅದರ ಸಹಿ ನಿಜವಾಗಿಯೂ ಅದರ
ವಿಷಯಗಳು ಮತ್ತು ಅದು ಹೊಂದುತಿರುವ ಕೀಲಿಯನ್ನು ಹೊಂದಿಸುತ್ತದೆ, ಆದ್ದರಿಂದ ಕೀಲಿಯನ್ನು ರಸೀದಿಯಿಂದ ಓದುವ ಪರಿಶೀಲಕ ಅದನ್ನು ಮಾನ್ಯವೆಂದು ವರದಿ ಮಾಡುತ್ತದೆ. ರಸೀದಿಯಲ್ಲಿ ಯಾವುದೂ ತಪ್ಪಾಗಿ ಕಾಣುವುದಿಲ್ಲ,
ಏಕೆಂದರೆ ಅದರಲ್ಲಿ ಯಾವುದೂ ವಿರೋಧಾತ್ಮಕತೆಯಿಲ್ಲ.


ಅದನ್ನು ಹಿಡಿಯುತ್ತದೆ ಎಂಬ ಒಂದು ಮಾತ್ರ ಕಾರಣವೆಂದರೆ ದಾಳಿಗಾರರ ಬಳಿ ಪ್ರಭಾವ ಬೀರುವ ಯಾವುದೇ ಮಾರ್ಗವಿಲ್ಲದ ಕೀಯನ್ನು ಪರಿಶೀಲಿಸುವುದು. 
ಕಾರಣದೊಂದಿಗೆ `verify_receipt` ಕೀಯನ್ನು ವಾದವಾಗಿವಾಗಿ ಪಡೆದುಕೊಳ್ಳುತ್ತದೆ: 
ಸರಿಯಾದ ಸುರಕ್ಷಿತ ರೀತಿಯಲ್ಲಿ ಬರೆಯಲಾಗದ ಫಂಕ್ಷನ್ ಇದರ ಸಹಿ `verify_receipt(receipt)` ಆಗಿದೆ, 
ಏಕೆಂದರೆ ಅದು ತಲುಪಬಹುದಾದ ಏಕೈಕ ಕೀ ಯಾರು ಕೂಡ ದಾಳಿಗಾರ ನೀಡಿದದ್ದು ಆಗಿರುತ್ತದೆ. 

ಅದೇ ರಸೀದಿ ನೀವು ನೋಟ್‌ಬುಕ್ ಹೊರಗೆ ಪರಿಶೀಲಿಸಲು ಬಯಸಿದರೆ `sample_receipts/04_resigned_receipt.json` ಆಗಿ ಉಳಿಸುತ್ತದೆ. 
ನೋಟ್ಬುಕ್‌ಗೆ ಹೊರಗಿನ ಅದನ್ನು ಪರಿಶೀಲಿಸಲು ಬಯಸುತ್ತೇನೆ.


## ಸೆಕ್ಷನ್ 3: ರಸೀದಿಗಳನ್ನು ಸರಪಳಿಯಾಗಿ ಜೋಡಿಸಿ

ಒಂದೊಂದು ರಸೀದಿ ಒಂದು ಕ್ರಿಯೆಯನ್ನು ರಕ್ಷಿಸುತ್ತದೆ. ಬಹುತೇಕ ಏಜೆಂಟ್‌ಗಳು ಅನೇಕ ಕ್ರಿಯೆಗಳು ಕೈಗೊಳ್ಳುತ್ತವೆ. ಸಂಪೂರ್ಣ ಕ್ರಮವನ್ನು ತಿದ್ದುಪಡಿಯಿಂದ ರಕ್ಷಿಸಲು, ನಾವು ಪ್ರತಿಯೊಂದು ಹೊಸ ರಸೀದಿಯPayloadನಲ್ಲಿ ಹಿಂದಿನ ರಸೀದಿಯ ಹ್ಯಾಶ್ ಅನ್ನು ಸೇರಿಸುವ ಮೂಲಕ ಪ್ರತಿಯೊಂದು ರಸೀದಿಯನ್ನು ಹಿಂತಿರುಗಿ ಲ್ಯಾಂಕ್ ಮಾಡುತ್ತೇವೆ.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

ಯಾರಾದರೂ ರಸೀದಿಯನ್ನು ಇಲ್ಲಿ ನೀಡಲಾದ ಕ್ರಮದ ಒಳಗೆ ತೆಗೆದುಹಾಕಿದರೆ ಅಥವಾ ಮರುಕ್ರಮ ಬೇರೆ ರೀತಿಯಲ್ಲಿ ಮಾಡಿದರೆ, ಮುಂದಿನ ಲಿಂಕ್ ಮುರಿದಿಹುದು. ಕಡಿತ ಅಥವಾ ಸೈನ್ ಮಾಡಿದ್ದವರು ನಿಯಂತ್ರಣದಲ್ಲಿರುವ ಪುಟಾಂತರದ ಪುನಃ ಬರವಣಿಗೆ ಪತ್ತೆ ಮಾಡಲು ಪಿನ್ ಮಾಡಲಾದ ಪ್ರಸ್ತುತ ತಲೆ, ಹೊರಗಿನ ಚೆಕ್ಪಾಯಿಂಟ್, ಅಥವಾ ಪ್ರಮಾಣೀಕೃತ ಜನಸಂಖ್ಯಾ ಕಟ್ಟಿಸ್ತಬುಧ ನಿರ್ವಹಣೆ ಬೇಕಾಗುತ್ತದೆ.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


ಈಗ ಮಧ್ಯದ ರಸೀದಿಯನ್ನು ಮಾರುತ ಮಾಡುವ ಮೂಲಕ ಸರಣಿಯನ್ನು ಬೆರಸಿ ಮತ್ತೆ ಪರಿಶೀಲಿಸಿ. ಮಾರುತಗೊಂಡ ರಸೀದಿ ತನ್ನ ಸಹಿ ಪರಿಶೀಲನೆಯಲ್ಲಿ ವಿಫಲವಾಗುತ್ತದೆ, ಮತ್ತು ಮುಂದಿನ ರಸೀದಿ ತನ್ನ ಸರಣಿ-ಲಿಂಕ್ ಪರಿಶೀಲನೆಯಲ್ಲಿ ವಿಫಲವಾಗುತ್ತದೆ (ಯಾಕೆಂದರೆ ಅದರ `previous_receipt_hash` ಸಂಶೋಧಿಸಲ್ಪಟ್ಟ ಮಧ್ಯದ ರಸೀದಿಯ ಹ್ಯಾಶ್‌ಗೆ ಇನ್ನಷ್ಟು ಹೊಂದಿಕೆಯಾಗುವುದಿಲ್ಲ).  


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


ರಸೀದು 0 ಇನ್ನೂ ಪರಿಶೀಲಿಸುತ್ತದೆ (அது மாற்றப்படவில்லை ಮತ್ತು ಯಾವುದೇ ಮುಂಚಿನವು ಅವಲಂಬಿಸಿಲ್ಲ). ರಸೀದು 1 ಅದರ ಸಹಿ ಪರೀಕ್ಷೆಯಲ್ಲಿ ವಿಫಲವಾಗಿದೆ ಏಕೆಂದರೆ ನಾವು `tool_args_hash` ಅನ್ನು ಬದಲಿಸಿದ್ದೇವೆ. ರಸೀದು 2 ತನ್ನ ಸರಪಳಿ-ಲಿಂಕ್ ಪರಿಶೀಲನೆಯಲ್ಲಿ ವಿಫಲವಾಗಿದೆ ಏಕೆಂದರೆ ಅದರ `previous_receipt_hash` ಮೂಲ (ಇப்போது ಬದಲಾಯಿಸಿದ) ರಸೀದು 1 ವಿರುದ್ಧ ಗಣನೆ ಮಾಡಲಾಗಿದೆ.

ಶತ್ರು ಬದಲಾಯಿಸಿದ ರಸೀದು 1 ಅನ್ನು ಮರು-ಸಹಿ ಮಾಡಿದರೂ ಸಹ, ಹೊಸ ಸಹಿ ಪ್ರಕಟಕನ ಕೀಲಿಯ ವಿರುದ್ಧ ವಿಫಲವಾಗುತ್ತದೆ, ಅವರು ಪ್ರಕಟಕನ ಖಾಸಗಿ ಕೀ ಹೊಂದಿರದಿದ್ದರೆ (ತಾವು ತಮ್ಮದೇ ಕೀಲಿಯೊಂದಿಗೆ ಮರು-ಸಹಿ ಮಾಡಿದರೆ ವಿಫಲವಾಗುತ್ತದೆ, ಸೆಕ್ಷನ್ 2 ನಲ್ಲಿ ತೋರಿಸಲ್ಪಟ್ಟದಂತೆ), ಮತ್ತು ರಸೀದು 2 ರಲ್ಲಿನ ಸರಪಳಿ-ಲಿಂಕ್ ಜೋಡಿ ತಪ್ಪು ಇನ್ನೂ ಹಂಚಿಕೆಯನ್ನು ಬಹಿರಂಗಪಡಿಸುವದು. ಬದಲಾವಣೆಯನ್ನು ಹಿಡಿದಿಡಲು, ಶತ್ರು ಬದಲಾವಣೆ ಬಿಂದುವಿನ ನಂತರ ಪ್ರತಿಯೊಂದು ರಸೀದಿನ ಮೇಲಿನ ಸಹಿಯನ್ನು ಮಾಡಬೇಕಾಗುತ್ತದೆ, ಇದು ಪ್ರಕಟಕನ ಖಾಸಗಿ ಕೀ ಹೊಂದಿರಬೇಕೆಂದು ಬಯಸುತ್ತದೆ.


## ಸೆಕ್ಷನ್ 4: ರಶೀದಿ ಸಹಿ ಸಹಿತ ಏಜೆಂಟ್ ಸಾಧನ ಕರೆ ಭಾರತದಿ

ನಿಜವಾದ ಜಾರಿಗೆ, ನೀವು ಪ್ರತಿಯೊಬ್ಬ ಏಜೆಂಟ್ ಲೇಖಕನೂ `make_receipt` ಅನ್ನು ಕರೆ ಮಾಡುವುದನ್ನು ನೆನಪಿಡಬೇಕಾಗಿಲ್ಲ. ಪ್ರತಿಯೊಂದು ಸಾಧನ ಕರೆಗೂ ರಶೀದಿ ಸಹಿ ಸ್ವಯಂಚಾಲಿತವಾಗಿರಬಹುದಾಗಿದೆ.

ಇಲ್ಲಿದೆ ಸರಳ ಮಾದರಿ: ಯಾವುದೇ ಕರೆಯಬಹುದಾದ ಸಾಧನ ಕಾರ್ಯವನ್ನು ತೆಗೆದುಕೊಳ್ಳುವ ಮತ್ತು ಅದರ ರಶೀದಿ ನಿರ್ಗಮಿಸುವ ಆವೃತ್ತಿಯನ್ನು ಹಿಂತಿರುಗಿಸುವ ಕವರ್ ಕ್ಲಾಸ್. ಇದು ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ (`agent_framework.foundry`) ಸೇರಿದಂತೆ ಯಾವುದೇ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ಗೆ ಸೂಕ್ತವಾಗುತ್ತದೆ.

ನೀವು ಮೈಕ್ರೋಸಾಫ್ಟ್ ಫೌಂಡ್ರಿ ಪ್ರಾಜೆಕ್ಟ್ ಸ್ಥಾಪಿಸದಿದ್ದರೆ, ಕೆಳಗಿನ ಸ್ಥಳೀಯ ಮಾದರಿಯು ಇನ್ನೂ ಆ ಮಾದರಿಯನ್ನು ಪ್ರದರ್ಶಿಸುತ್ತದೆ.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್‌ ಜೊತೆಗೆ ಏಕರೂಪಗೊಳಿಸುವುದು

ಮೇಲಿನ `ReceiptedTool` ರ್ಯಾಪರ್ ಫ್ರೇಮ್ವರ್ಕ್-ಸ್ವತಂತ್ರವಾಗಿದೆ. ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ ಬಳಸಿ ನಿರ್ಮಿಸಲಾದ ಏಜೆಂಟ್‌ನೊಳಗೆ ಅದನ್ನು ಬಳಸಲು, ರ್ಯಾಪ್ ಮಾಡಲ್ಪಟ್ಟ ಕಾರ್ಯಚಟುವಟಿಕೆಯನ್ನು ಉಪಕರಣವಾಗಿ ನೋಂದಾಯಿಸಿ. ಒಂದು ರೇಖಾಚಿತ್ರ (ನೀವು ಮಾಕ್ ಅನ್ನು ನೈಜ ಮೈಕ್ರೋಸಾಫ್ಟ್ ಫೌಂಡ್ರಿ ಉಪಕರಣ ನೋಂದಣಿಯಿಂದ ಬದಲಾಯಿಸಬಹುದು):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್‌ಕ್ಕೆ ರಸೀದಿ ಬಗ್ಗೆ ಏನೂ ತಿಳಿಯಬೇಕಾಗಿಲ್ಲ. ರಸೀದಿ ಸಹಿಮೋಹನವು ಉಪಕರಣದ ಸುತ್ತಲೂ ರ್ಯಾಪ್ ಆಗಿದೆ, ಫ್ರೇಮ್ವರ್ಕ್‍ಗೆ ಸೇರಿಸಲ್ಪಟ್ಟಿಲ್ಲ. ಇದುವೇ ಏಜೆಂಟ್ ಕೋಡ್ ಅನ್ನು ಮರುಬರಹಿಸದೆ provenance (ಮೂಲಸ್ಥಾನ) ಸೇರಿಸುವ ವಿಧಾನ.


## ಮರುಸರಣಿ ಮತ್ತು ಮಟ್ಟದ ಸವಾಲು

ನೀವು:

- Ed25519 ಕೀ ಜೋಡಿ 생성 ಮಾಡಿದ್ದು.
- ಏಜೆಂಟ್ ಟೂಲ್ ಕರೆಯುವಿಕೆಗೆ ರಸೀದಿ ರಚಿಸಿ ಸ್ವೀಕರಿಸಿಕೊಂಡಿದ್ದು.
- ರಸೀದಿಯನ್ನು ಆಫ್‌ಲೈನ್‌ನಲ್ಲಿ ಕೇವಲ ಪ್ರಕಟಕుడಿನ ಸಾರ್ವಜನಿಕ ಕೀಪಾಯಿಂಟ್ ಬಳಸಿ, ರಸೀದಿಗೆ ಸ್ವತಂತ್ರವಾಗಿ ಪಡೆದು ಪರಿಶೀಲಿಸಿದೆ.
- ರಸೀದಿಯನ್ನು ತೊಂದರೆಯಾದಂತೆ ಮಾಡಿದಾಗ ಪರಿಶೀಲನೆಯ ವೈಫಲ್ಯವನ್ನು ಗಮನಿಸಿದರು.
- ಮೂರೂ ರಸೀದಿಗಳ ಹ್ಯಾಶ್-ಚೈನ್ ಕ್ರಮವನ್ನು ನಿರ್ಮಿಸಿದ್ದಾರೆ.
- ಸರಣಿಯ ಮಧ್ಯಭಾಗ ತೊಂದರೆಯಲ್ಲಿ ಇದ್ದಾಗ ಸಹಿ ವೈಫಲ್ಯ ಮತ್ತು ಸರಣಿ-ಕಡತ ವೈಫಲ್ಯ ಎರಡನ್ನೂ ಗಮನಿಸಿದರು.
- ಸ್ವಯಂಚಾಲಿತವಾಗಿ ರಸೀದಿ ಸಹಿ ಮಾಡುವ ಟೂಲ್ ಕಾರ್ಯವನ್ನು ಮುಚ್ಚಿದ್ದಾರೆ.

**ಮಟ್ಟದ ಸವಾಲು.** ರಸೀದಿ ನಮೂನೆಯಲ್ಲಿ `request_id` ಕ್ಷೇತ್ರವನ್ನು (ವಿತರಣಾ ಟ್ರೇಸಿಂಗ್ ಗಾಗಿ UUID) ವಿಸ್ತರಿಸಿ. `make_receipt` ನ್ನು ಇದನ್ನು ಸೇರಿಸಿ, ಮತ್ತು ರಸೀದಿಗಳನ್ನು ಮುಗಿಸು ಮುನ್ಸೂಚನೆ ಅಂತಿಮ ಪರೀಕ್ಷೆ ಮಾಡಿ. ನಂತರ ಸಹಿ ಮಾಡಿದ ಬಳಿಕ ಕ್ಷೇತ್ರವನ್ನು ಬದಲಿಸಿ ಪರಿಶೀಲನೆ ವೈಫಲ್ಯವಾಗುವುದನ್ನು ಖಚಿತಪಡಿಸಿ. ಇದು ಪ್ರತಿಯೊಂದು ಬೈಟ್ canonical ಎನ್ಕೋಡಿಂಗ್ ಸಹಿಗೆ ಹೇಗೆ ಸಹಕರಿಸುತ್ತದೆ ಎಂದು ಅರ್ಥಮಾಡಿಕೊಳ್ಳಲು ಪ್ರೇರಣೆ ನೀಡುತ್ತದೆ.

**ಮುಖ್ಯ ವ್ಯತ್ಯಾಸ.** ಸ್ಥಿರ ಕೀ ಮತ್ತು ಸರಣಿ ಪ್ರಿಂಕಾರದಲ್ಲಿ, ರಸೀದಿಗಳು ಸಹಿ ಸಂಬಂಧ, ವಿಷಯ ಸಮಗ್ರತೆ ಮತ್ತು ಮುಂಚೆಯುತ್ತಿರುವ ಲಿಂಕ್ ಅನ್ನು ನೆನಪಿಸುವುದನ್ನು ಸಾಬೀತು ಮಾಡುತ್ತವೆ. ಅವು ಕಾರ್ಯಚರಣೆ, ಮೂಲ ಸತ್ಯ, ನಂಬಬಹುದಾದ ಸಮಯ, ಪೂರ್ಣತೆ, ನೀತಿ ದರ್ಶನ ಅಥವಾ ಸರಿಯಾದಿಕೆಯನ್ನು ಸಾಬೀತು ಮಾಡವುದಿಲ್ಲ. ರಸೀದಿಗಳು ಆಧಾರಧಾರಣೆಯಾಗಿದೆ. ಆಡಳಿತವನ್ನು ನೀವು ಮೇಲೊಂದಿಗಿರಿಸಿದ ವ್ಯವಸ್ಥೆಯಾಗಿ ನಿರ್ಮಿಸುವಿರಿ.

ಆಪ್ತ ನಿರ್ದಿಷ್ಟತೆಗೆ ಆ ಪಾಠದ README ಅನ್ನು ಮತ್ತೆ ಓದಿ. ರಸೀದಿಗಳಲ್ಲಿ ತಂಡಗಳು ಸಾಮಾನ್ಯವಾಗಿ ಮಾಡುವ ತಪ್ಪು "ನಾವು ರಸೀದಿಗಳನ್ನು ಹೊಂದಿದ್ದೇವೆ" ಅಂದರೆ "ನಾವು ಆಡಳಿತದಲ್ಲಿದ್ದೇವೆ" ಎಂದು ಊಹಿಸುವುದು. ಅದು ಅಲ್ಲ. ರಸೀದಿಗಳು ಏಜೆಂಟ್ ನಡೆಸಲಾದ ಕ್ರಮವನ್ನು ಪರಿಶೀಲಿಸಲು ಸಹಾಯ ಮಾಡುತ್ತವೆ. ಅವು ಅದನ್ನು ಸರಿಯಾಗೆ ಮಾಡುತ್ತವೆ ಅಂದ್ರೆ ಅಲ್ಲ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
